# LF7 — vision_model: bỏ phiếu độ hữu dụng bằng Claude vision

Đây là **một labeling function (LF7)** trong khung weak-supervision (xem `docs/Labeling_Plan.md`),
độ tin **trung bình** — KHÔNG phải nhãn cuối. Nhãn cuối do label model hợp nhất LF1–LF10; tín hiệu
mạnh nhất là **correctness vs GT** (LF1–5, `notebooks/label_correctness.ipynb`).

## 5 tác vụ theo required-view (xem paper §3.2)
Hữu dụng = thấy rõ đối tượng đủ để **đánh giá** (kể cả kết luận khỏe mạnh), KHÔNG đòi triệu chứng.
- `1_maturity_evaluation` — độ chín: thấy rõ **trái** (màu/hình/kích thước vỏ trung thực).
- `2_foliar_disease` — bệnh lá (Gray Leaf Spot, Leaf Rot): thấy rõ **phiến lá** đủ đánh giá.
- `3_trunk_disease` — bệnh thân (Stem Bleeding): thấy rõ **bề mặt thân**.
- `4_crown_disease` — bệnh đọt/crown (Bud Rot): thấy rõ **đỉnh đọt** (nhìn từ trên xuống).
- `5_petiole` — tình trạng tàu lá qua **cuống lá / độ rủ** (Bud Root Dropping): thấy lờ mờ độ rủ cũng đủ.

## LF7 bỏ phiếu mỗi tác vụ: `1` / `0` / `abstain`
`conf ≥ τ_high → 1` (hữu dụng), `conf ≤ τ_low → 0` (không), giữa → `abstain` (để LF khác/label model quyết).

Tầng 1 (cổng chất lượng tất định) chạy trước — ảnh rớt cổng: LF7 abstain toàn bộ (thuộc miền chất lượng).

## 1. API key

In [9]:
# %pip install anthropic pillow pandas numpy
import os, subprocess
# Nạp key từ macOS Keychain nếu chưa có trong biến môi trường
if not os.getenv('ANTHROPIC_API_KEY'):
    os.environ['ANTHROPIC_API_KEY'] = subprocess.check_output(
        ['security', 'find-generic-password', '-s', 'ANTHROPIC_API_KEY', '-w']).decode().strip()
assert os.environ.get('ANTHROPIC_API_KEY'), 'Chưa set ANTHROPIC_API_KEY'

## 2. Cấu hình

In [10]:
import base64, io
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image, ImageOps
import anthropic

ROOT = Path.cwd()
if not (ROOT/'Dataset').exists():
    for p in ROOT.parents:
        if (p/'Dataset').exists(): ROOT=p; break
OUT = ROOT/'labels'; OUT.mkdir(exist_ok=True)

MODEL='claude-sonnet-5'; MAX_LONG_EDGE=1024; N_PER_SOURCE=8   # pilot rẻ: đổi lại 'claude-opus-4-8' & tăng N cho mẻ đầy đủ
TAU_HIGH, TAU_LOW = 0.70, 0.30      # ngưỡng vote của LF7 (1 / abstain / 0)

# cổng chất lượng (tất định)
GATE_PROC_DIM=512; GATE_MIN_DIM=200; GATE_BLUR_MIN=60.0
GATE_DARK_MAX=0.60; GATE_BRIGHT_MAX=0.45; GATE_MEAN_MIN=25; GATE_MEAN_MAX=235

# 5 tác vụ, đúng thứ tự AGENTS.md
TASKS=['1_maturity_evaluation','2_foliar_disease','3_trunk_disease','4_crown_disease','5_petiole']  # nhãn chuẩn (khớp label_correctness.ipynb)
# pydantic không cho định danh bắt đầu bằng số -> ánh xạ nhãn chuẩn sang tên trường của mô hình:
FIELD={'1_maturity_evaluation':'maturity_evaluation','2_foliar_disease':'foliar_disease','3_trunk_disease':'trunk_disease','4_crown_disease':'crown_disease','5_petiole':'petiole'}
IMG_EXTS={'.jpg','.jpeg','.png','.bmp','.webp'}
# ánh xạ thư mục -> tác vụ native (LF8, để đối chiếu)
FOLDER_NATIVE={'Gray Leaf Spot':{'2_foliar_disease'},'Leaf Rot':{'2_foliar_disease'},
  'Stem Bleeding':{'3_trunk_disease'},'Bud Rot':{'4_crown_disease'},
  'Bud Root Dropping':{'5_petiole'},'coconut-veirf-v5':{'1_maturity_evaluation'}}
client=anthropic.Anthropic()

## 3. Tầng 1 — cổng chất lượng tất định (miễn phí)

In [11]:
_LAP=np.array([[0,1,0],[1,-4,1],[0,1,0]],dtype=np.float32)
def _lapvar(g):
    p=np.pad(g,1,mode='reflect'); out=np.zeros_like(g)
    for dy in range(3):
        for dx in range(3): out+=_LAP[dy,dx]*p[dy:dy+g.shape[0],dx:dx+g.shape[1]]
    return float(out.var())
def quality_gate(path):
    img=ImageOps.exif_transpose(Image.open(path)).convert('RGB'); w,h=img.size
    s=GATE_PROC_DIM/max(w,h) if max(w,h)>GATE_PROC_DIM else 1.0
    small=img.resize((max(1,int(w*s)),max(1,int(h*s)))) if s<1 else img
    arr=np.asarray(small,dtype=np.float32); g=arr@np.array([.299,.587,.114],dtype=np.float32)
    lap,mean=_lapvar(g),float(g.mean()); dark,bright=float((g<15).mean()),float((g>245).mean())
    r=[]
    if min(w,h)<GATE_MIN_DIM: r.append('lowres')
    if lap<GATE_BLUR_MIN: r.append('blur')
    if dark>GATE_DARK_MAX or mean<GATE_MEAN_MIN: r.append('underexposed')
    if bright>GATE_BRIGHT_MAX or mean>GATE_MEAN_MAX: r.append('overexposed')
    return {'quality_pass':not r,'gate_reason':'|'.join(r),'lap_var':round(lap,1),'mean_bright':round(mean,1),'min_dim':min(w,h)}

## 4. Rubric + schema (5 tác vụ)

In [12]:
SYSTEM='''Bạn là giám định viên ảnh nông nghiệp. Với ảnh cây/trái dừa (đã qua bộ lọc chất lượng cơ bản),
hãy đánh giá ĐỘC LẬP cho từng tác vụ: ảnh có cho thấy rõ đối tượng/bộ phận cần thiết ĐỦ ĐỂ ĐÁNH GIÁ tác vụ đó hay không.
NGUYÊN TẮC: suitable=true khi ảnh đủ rõ để KẾT LUẬN — dù kết luận là CÓ hay KHÔNG có vấn đề. KHÔNG đòi hỏi ảnh phải đang có triệu chứng bệnh: một bộ phận KHỎE MẠNH nhưng hiện rõ, đủ nét vẫn suitable=true. Chỉ đặt suitable=false khi đối tượng cần thiết không hiện diện, bị che khuất, hoặc quá kém để đánh giá.
- maturity_evaluation: thấy rõ >=1 trái dừa với kích thước/hình dạng/màu vỏ tái hiện trung thực, đủ để đánh giá độ chín (dry/green/tender).
- foliar_disease: thấy rõ phiến lá/tàu lá đủ nét để đánh giá tình trạng mặt lá (có hay không đốm/cháy/thối).
- trunk_disease: thấy rõ bề mặt thân/gốc đủ nét để đánh giá tình trạng thân (có hay không chảy nhựa/loét/đổi màu).
- crown_disease: thấy rõ đỉnh đọt/ngọn (crown), lý tưởng nhìn từ trên xuống, đủ để đánh giá tình trạng đọt (có hay không thối/héo/đổi màu).
- petiole: thấy được CUỐNG LÁ dừa — đoạn thân giữa của tàu lá: trơn, dài (nằm giữa bẹ ôm thân và đoạn sống lá mang lá chét) — hoặc tư thế/độ rủ của nó (vươn thẳng hay rủ xuống/gãy gập), đủ để nhận định độ rủ/vàng úa/còi cọc (hay khỏe mạnh). Độ rủ là dấu hiệu thô, thấy lờ mờ cũng đủ.
Nếu đối tượng cần thiết không hiện diện hoặc bị che khuất/kém quá mức cho một tác vụ -> suitable=false.
confidence là số thực 0..1 = mức độ ảnh ĐỦ RÕ để đánh giá tác vụ đó (KHÔNG phải xác suất có bệnh).'''
from pydantic import BaseModel
class Task(BaseModel):
    suitable: bool; confidence: float; reason: str
class ImageLabel(BaseModel):
    maturity_evaluation: Task; foliar_disease: Task; trunk_disease: Task; crown_disease: Task; petiole: Task

## 5. Lấy mẫu

In [13]:
def collect():
    d0=ROOT/'Dataset'/'Coconut Tree Disease Dataset'
    for f in ['Gray Leaf Spot','Leaf Rot','Stem Bleeding','Bud Rot','Bud Root Dropping']:
        d=d0/f
        if d.is_dir(): yield f, sorted(p for p in d.iterdir() if p.suffix.lower() in IMG_EXTS)
    imgs=[]
    for sp in ['train','valid','test']:
        d=ROOT/'Dataset'/'coconut-veirf-v5'/sp/'images'
        if d.is_dir(): imgs+=sorted(p for p in d.iterdir() if p.suffix.lower() in IMG_EXTS)
    yield 'coconut-veirf-v5', imgs
def sample_even(a,n):
    if len(a)<=n: return a
    st=len(a)/n; return [a[int(i*st)] for i in range(n)]
pilot=[(p,f) for f,imgs in collect() for p in sample_even(imgs,N_PER_SOURCE)]
print(len(pilot),'ảnh'); pd.Series([f for _,f in pilot]).value_counts()

48 ảnh


Gray Leaf Spot       8
Leaf Rot             8
Stem Bleeding        8
Bud Rot              8
Bud Root Dropping    8
coconut-veirf-v5     8
Name: count, dtype: int64

## 6. Gọi model

In [14]:
def encode(path):
    img=ImageOps.exif_transpose(Image.open(path)).convert('RGB'); w,h=img.size
    if max(w,h)>MAX_LONG_EDGE: s=MAX_LONG_EDGE/max(w,h); img=img.resize((int(w*s),int(h*s)))
    b=io.BytesIO(); img.save(b,format='JPEG',quality=85); return base64.standard_b64encode(b.getvalue()).decode()
def label_image(path):
    return client.messages.parse(model=MODEL,max_tokens=1024,system=SYSTEM,
        output_config={'effort':'medium'},
        messages=[{'role':'user','content':[
            {'type':'image','source':{'type':'base64','media_type':'image/jpeg','data':encode(path)}},
            {'type':'text','text':'Đánh giá độc lập ảnh dừa này cho từng tác vụ.'}]}],
        output_format=ImageLabel).parsed_output
def vote(conf):
    return 1 if conf>=TAU_HIGH else (0 if conf<=TAU_LOW else -1)   # -1 = abstain

## 7. Chạy — cổng trước, LF7 vote sau

In [15]:
rows=[]
for i,(path,folder) in enumerate(pilot,1):
    q=quality_gate(path)
    row={'image_id':path.stem,'source_folder':folder,'path':str(path.relative_to(ROOT)),**q}
    if q['quality_pass']:
        try: r=label_image(path)
        except Exception as e: print('ERR',path.name,e); continue
        for t in TASKS:
            tk=getattr(r,FIELD[t])
            row[f'lf7_{t}']=vote(tk.confidence)          # phiếu LF7: 1/0/-1(abstain)
            row[f'{t}_conf']=round(tk.confidence,3)
            row[f'{t}_reason']=tk.reason
    else:
        for t in TASKS:
            row[f'lf7_{t}']=-1; row[f'{t}_conf']=np.nan; row[f'{t}_reason']=''  # rớt cổng -> abstain
    rows.append(row)
    if i%10==0 or i==len(pilot): print(f'  {i}/{len(pilot)}')
df=pd.DataFrame(rows); df.to_csv(OUT/'lf7_vision_pilot.csv',index=False)
print('Đã ghi',OUT/'lf7_vision_pilot.csv')
df[['image_id','source_folder','quality_pass']+[f'lf7_{t}' for t in TASKS]].head(10)

  10/48
  20/48
  30/48
  40/48
  48/48
Đã ghi /Users/peggy/Documents/Projects/HK2/coconut-iqa/labels/lf7_vision_pilot.csv


,image_id,source_folder,quality_pass,lf7_1_maturity_evaluation,lf7_2_foliar_disease,lf7_3_trunk_disease,lf7_4_crown_disease,lf7_5_petiole
0,GrayLeafSpot001,Gray Leaf Spot,True,0,1,0,0,0
1,GrayLeafSpot1151,Gray Leaf Spot,True,0,1,0,0,0
2,GrayLeafSpot1394,Gray Leaf Spot,True,0,1,0,0,1
3,GrayLeafSpot1637,Gray Leaf Spot,True,0,1,0,0,0
4,GrayLeafSpot188,Gray Leaf Spot,True,0,0,0,0,1
5,GrayLeafSpot2122,Gray Leaf Spot,True,0,1,0,0,0
6,GrayLeafSpot466,Gray Leaf Spot,True,0,1,0,0,0
7,GrayLeafSpot733,Gray Leaf Spot,True,0,1,0,0,-1
8,LeafRot001,Leaf Rot,True,0,1,0,0,-1
9,LeafRot110,Leaf Rot,True,0,1,0,0,0


## 8. Kiểm tra LF7 vs thư mục native (LF8)

In [16]:
passed=df[df['quality_pass']]
print(f'Tổng {len(df)} | rớt cổng {int((~df.quality_pass).sum())} | qua cổng {len(passed)}')
print('\nPhiếu LF7 mỗi tác vụ (1=hữu dụng / 0=không / -1=abstain):')
for t in TASKS:
    vc=passed[f'lf7_{t}'].value_counts().to_dict()
    print(f'  {t:22s}: 1={vc.get(1,0)}  0={vc.get(0,0)}  abstain={vc.get(-1,0)}')
# khớp native: trên tác vụ native của thư mục, LF7 vote 1?
print('\nLF7 vote=1 trên tác vụ NATIVE của thư mục (mong đợi cao):')
for folder in passed['source_folder'].unique():
    sub=passed[passed.source_folder==folder]; nat=list(FOLDER_NATIVE[folder])[0]
    pos=int((sub[f'lf7_{nat}']==1).sum())
    print(f'  {folder:18s} -> {nat:22s}: {pos}/{len(sub)}')
print('\nCa cần review (qua cổng, native nhưng LF7 KHÔNG vote 1):')
for _,r in passed.iterrows():
    nat=list(FOLDER_NATIVE[r.source_folder])[0]
    if r[f'lf7_{nat}']!=1:
        print(f"  {r.source_folder:16s} {r.image_id:20s} {nat}: vote={r[f'lf7_{nat}']} conf={r[f'{nat}_conf']}")

Tổng 48 | rớt cổng 0 | qua cổng 48

Phiếu LF7 mỗi tác vụ (1=hữu dụng / 0=không / -1=abstain):
  1_maturity_evaluation : 1=11  0=35  abstain=2
  2_foliar_disease      : 1=29  0=17  abstain=2
  3_trunk_disease       : 1=10  0=32  abstain=6
  4_crown_disease       : 1=7  0=30  abstain=11
  5_petiole             : 1=16  0=16  abstain=16

LF7 vote=1 trên tác vụ NATIVE của thư mục (mong đợi cao):
  Gray Leaf Spot     -> 2_foliar_disease      : 7/8
  Leaf Rot           -> 2_foliar_disease      : 8/8
  Stem Bleeding      -> 3_trunk_disease       : 8/8
  Bud Rot            -> 4_crown_disease       : 3/8
  Bud Root Dropping  -> 5_petiole             : 3/8
  coconut-veirf-v5   -> 1_maturity_evaluation : 8/8

Ca cần review (qua cổng, native nhưng LF7 KHÔNG vote 1):
  Gray Leaf Spot   GrayLeafSpot188      2_foliar_disease: vote=0 conf=0.2
  Bud Rot          BudRot001            4_crown_disease: vote=-1 conf=0.6
  Bud Rot          BudRot059            4_crown_disease: vote=-1 conf=0.55
  Bud Rot    